In [63]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

PROPERTIES = 'global_properties.pkl'
GLOBAL_MAP = 'AUG_global_map.pkl'
POLYGON_DB = 'epos_cache.pkl'
FINAL_SOLUTION = 'optimized_solution_fast.pkl'
GRAND_COURIER = 'courier_training_checkpoint.pkl'


with open(PROPERTIES, 'rb') as f:
    properties_global = pickle.load(f)

with open(GLOBAL_MAP, 'rb') as f:
    AUG_global_map = pickle.load(f)

with open(POLYGON_DB, 'rb') as f:
    EPOS_results = pickle.load(f)

with open(FINAL_SOLUTION, 'rb') as f:
    final_solution = pickle.load(f)

with open(GRAND_COURIER, 'rb') as f:
    trainer = pickle.load(f)

properties = {}
for polygon, props in properties_global.items():
    if polygon in EPOS_results.keys():
        properties[polygon] = props
    else:
        continue

In [64]:
centroid_coords = np.array([
    [prop['centroid_lat'], prop['centroid_lon']] 
    for prop in properties.values()
])
polygon_ids = list(properties.keys())
spatial_tree = cKDTree(centroid_coords)

def save_check(FILE_NAME, datacache):
    with open(FILE_NAME, 'wb') as f:
        pickle.dump(datacache, f)

def get_nearest_polygons(lat, lon, k=15):
    """Get k nearest polygons using KD-Tree"""
    distances, indices = spatial_tree.query([lat, lon], k=min(k, len(polygon_ids)))
    return [polygon_ids[i] for i in indices]

def get_EPOS_strategy(results):
    """Safely compute strategy from EPOS results"""
    strategy_dict = {
        'christofides': 0,
        'BC': 0,
        'LKH3': 0,
        'Greedsearch': 0
    }

    poly_strat = {
        'christofides': 0,
        'BC': 0,
        'LKH3': 0,
        'Greedsearch': 0
    }

    poly_strategy = {}
    
    for polygon_id, item in results.items():
        if 'algorithm_wins' in item and len(item['algorithm_wins']) == 4:

            # polygon-specific strategy
            poly_strat['christofides'] = int(item['algorithm_wins'][0])
            poly_strat['BC'] = int(item['algorithm_wins'][1])
            poly_strat['LKH3'] = int(item['algorithm_wins'][2])
            poly_strat['Greedsearch'] = int(item['algorithm_wins'][3])
            
            # global average strategy
            strategy_dict['christofides'] += int(item['algorithm_wins'][0])
            strategy_dict['BC'] += int(item['algorithm_wins'][1])
            strategy_dict['LKH3'] += int(item['algorithm_wins'][2])
            strategy_dict['Greedsearch'] += int(item['algorithm_wins'][3])
        elif 'chosen_algorithm' in item:
            # Fallback: count chosen algorithm
            algo = item['chosen_algorithm'].lower()
            if 'christofides' in algo:
                strategy_dict['christofides'] += 1
                poly_strat['christofides'] += 1
            elif 'branch' in algo or 'cut' in algo:
                strategy_dict['BC'] += 1
                poly_strat['BC'] += 1
            elif 'lkh' in algo:
                strategy_dict['LKH3'] += 1
                poly_strat['LKH3'] += 1
            else:
                strategy_dict['Greedsearch'] += 1
                poly_strat['Greedsearch'] += 1

        total_poly_obs = max(1, sum(poly_strat.values()))
        poly_strategy[polygon_id] = [item / total_poly_obs for item in poly_strat.values()]
        
    total_obs = max(1, sum(strategy_dict.values()))
    global_cluster_strategy = [item / total_obs for item in strategy_dict.values()]
    
    return global_cluster_strategy, poly_strategy

def get_exp_couriers(max_couriers=10, checkpoint_every=2, dynamic_config = None):
    if dynamic_config is None:
        dynamic_config = {'U': 9000,
                          'price': 5000,
                          'CT_SA': 2,
                          'time_window': (0.0,10000.0)
                         }
    
    CHECKPOINT_FILE = 'NN_training_checkpoint.pkl'
    
    # Load checkpoint if exists
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'rb') as f:
            trained_couriers = pickle.load(f)
        start_idx = len(trained_couriers)
        print(f"Resumed from checkpoint: {start_idx} couriers trained")
    else:
        trained_couriers = {}
        start_idx = 0

    courier_list = []
    for courier, parameters in final_solution.items():
        if len(courier_list) >= max_couriers:
            break
        elif parameters['New route'] == [0]:
            continue
        else:
            courier_list.append(courier)
    
    for courier in tqdm(courier_list):

        lat = 55.752
        lon = 37.6177
        
        poly_harv_dist_sorted = get_nearest_polygons(lat, lon, k=1000)
        
        # Run EPOS with caching
        EPOS_temp = {}
        for pol, props in EPOS_results.items():
            if pol in poly_harv_dist_sorted:
                EPOS_temp[pol] = props
            else:
                continue
        
        # Store only strategy (not full results)
        trained_couriers[courier] = {
            'base_lat': lat,
            'base_lon': lon,
            'polygon_list': poly_harv_dist_sorted,
            'EPOS_global_strat': get_EPOS_strategy(EPOS_temp)[0],
            'EPOS_local_strat': get_EPOS_strategy(EPOS_temp)[1]
        }
        
        # Checkpoint
        if len(trained_couriers) % checkpoint_every == 0:
            save_check(CHECKPOINT_FILE, trained_couriers)
            print(f"Checkpoint saved: {len(trained_couriers)} couriers")
    
    # Final save
    save_check(CHECKPOINT_FILE, trained_couriers)

    return trained_couriers

In [65]:
trained_couriers = get_exp_couriers(1, 1)

Resumed from checkpoint: 10 couriers trained


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.94it/s]

Checkpoint saved: 10 couriers


In [66]:
len(trained_couriers[35]['polygon_list'])

1000

In [67]:
for item in properties.keys():
    print (AUG_global_map[item].keys())
    break

dict_keys(['points cloud', 'full map', 'distance matrix', 'augmented', 'cluster'])


In [78]:
class PolygonAlgNN(nn.Module):
    
    def __init__(self, input_dim=11, hidden_dim=128, classes=4):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.15),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.15),
            
            nn.Linear(hidden_dim, classes),
        )
    
    def forward(self, x):
        return self.network(x)

def prepare_courier_training_data(trained_couriers, properties, AUG_global_map):
    """
    Convert EPOS results into supervised learning dataset
    """
    
    courier_ids = []
    training_data = {}
    scalers = {}
    
    for courier_id, courier_data in trained_couriers.items():
        scaler = StandardScaler()
        training_data[courier_id] = {
            'X': [], 
            'y': [],
            'scaler': scaler
        } 
        X_list = []  # Polygon features
        y_list = []  # Algorithm policies (soft labels)
        for polygon_id in courier_data['polygon_list']:
            # Extract polygon features (from properties or global_cache)
            if polygon_id in properties:
                props = properties[polygon_id]
                params = AUG_global_map[polygon_id]
                avg_dist = np.mean([np.mean(dist_list) for dist_list in params['distance matrix']])
                features = [
                    props['area_km2'],
                    props['centroid_lat'],
                    props['centroid_lon'],
                    props['bounding_box']['width_km'],
                    props['bounding_box']['height_km'],
                    props['relative_position']['normalized_lat'],
                    props['relative_position']['normalized_lon'],
                    props['relative_position']['distance_from_center_km'],
                    avg_dist,
                    props['num_points'],
                    params['cluster']/7.0
                ]
            else:
                continue

# road distribution metrics and diversity
            
            # Extract policy from EPOS_local_strat (soft labels)
            if polygon_id in courier_data['EPOS_local_strat']:
                policy = courier_data['EPOS_local_strat'][polygon_id]
            else:
                policy = courier_data['EPOS_global_strat']
                
            X_list.append(features)
            policy = np.asarray(policy)
            policy = np.clip(policy, 1e-8, 1)
            policy_norm = policy/policy.sum()
            y_list.append(policy_norm)

        X_array = np.array(X_list)
        print(np.shape(X_array))
        features_norm = scaler.fit_transform(X_array)
        
        training_data[courier_id] = {
            'X': features_norm, 
            'y': np.array(y_list),
            'scaler': scaler
        }

        scalers[courier_id] = scaler
    
    return training_data, scalers

def train_personalized_models(courier_datasets, checkpoint_dir='courier_models', 
                               num_epochs=500, val_split=0.2, patience=100):
    """Train with robust early stopping for small datasets"""

    trained_models = {}
    
    for courier_id, data in tqdm(courier_datasets.items(), desc="Training"):
        X = data['X']
        y = data['y']
        scaler = data['scaler']
        
        # Train/val split
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=val_split, random_state=42
        )
        
        # Convert to tensors
        X_train_tensor = torch.tensor(X_train, dtype=torch.float)
        y_train_tensor = torch.tensor(y_train, dtype=torch.float)
        X_val_tensor = torch.tensor(X_val, dtype=torch.float)
        y_val_tensor = torch.tensor(y_val, dtype=torch.float)
        
        # DataLoader
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_dataset, batch_size=min(16, len(X_train)), shuffle=True)
        
        # Model & optimizer
        model = PolygonAlgNN(input_dim=X_train.shape[1])
        optimizer = torch.optim.Adam(model.parameters(), lr=0.015, weight_decay = 1e-4)  # Reduced from 0.01
        criterion = nn.KLDivLoss(reduction='batchmean')
        
        # Learning rate scheduler (alternative to early stopping)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)
        
        # Training with robust early stopping
        learning_perf = {'train': [], 'val': []}
        best_val_loss = float('inf')
        no_improve = 0
        min_epochs = patience  # Train at least 80 epochs before considering early stop
        best_model_state = None
        
        model.train()
        for epoch in range(num_epochs):
            # Training step
            epoch_train_loss = 0
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                logits = model(batch_X)
                output = F.log_softmax(logits, dim=1)
                loss = criterion(output, batch_y)
                loss.backward()
                optimizer.step()
                epoch_train_loss += loss.item()
            
            # Validation step
            model.eval()
            with torch.no_grad():
                val_output = F.log_softmax(model(X_val_tensor), dim=1)
                val_loss = criterion(val_output, y_val_tensor).item()
            
            learning_perf['train'].append(epoch_train_loss / len(train_loader))
            learning_perf['val'].append(val_loss)
            
            # Update scheduler
            scheduler.step()
            
            # Robust early stopping (only after min_epochs)
                # Use smoothed validation loss (5-epoch moving average)
            recent_vals = learning_perf['val'][-5:] if len(learning_perf['val']) >= 5 else learning_perf['val']
            val_loss_smooth = np.mean(recent_vals)
                
                # Only count as improvement if decrease is significant
            if val_loss_smooth < best_val_loss - 1e-4:
                best_val_loss = val_loss_smooth
                no_improve = 0
                best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience and epoch >= min_epochs:
                    print(f"Courier {courier_id}: Early stop at epoch {epoch+1} (val_loss={val_loss:.4f})")
                    break
            
            model.train()
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            
        # Save best model
        model_path = os.path.join(checkpoint_dir, f'courier_{courier_id}_model.pth')
        torch.save({
            'model_state_dict': model.state_dict(),
            'input_dim': X_train.shape[1],
            'n_samples': len(X_train),
            'best_val_loss': best_val_loss,
            'epochs_trained': epoch + 1,
            'scaler_mean': scaler.mean_,
            'scaler_scale': scaler.scale_,
        }, model_path)
        
        trained_models[courier_id] = {
            'path': model_path,
            'n_samples': len(X_train),
            'best_val_loss': best_val_loss,
            'epochs_trained': len(learning_perf['train']),
            'scaler_mean': scaler.mean_,
            'scaler_scale': scaler.scale_,
        }
        print(f"Courier {courier_id}: Trained {epoch+1} epochs, Val Loss={best_val_loss:.4f}")

    return trained_models

In [79]:
# === PIPELINE ===
def run_nn_pipeline(trained_couriers, properties, AUG_global_map, checkpoint_dir='courier_models'):
    """Complete NN training pipeline"""
    print("=== Step 1: Preparing Training Data ===")
    courier_datasets, scalers = prepare_courier_training_data(trained_couriers, properties, AUG_global_map)
    print(f"Prepared datasets for {len(courier_datasets)} couriers")
    
    print("\n=== Step 2: Training Personalized Models ===")
    trained_models = train_personalized_models(courier_datasets, checkpoint_dir=checkpoint_dir)
    print(f"Trained {len(trained_models)} models")
    
    print("\n=== Step 3: Saving Results ===")
    print(f"Models saved to: {checkpoint_dir}")
    
    return trained_models, courier_datasets

In [80]:
run_nn_pipeline(trainer, properties_global, AUG_global_map)

=== Step 1: Preparing Training Data ===
(2000, 11)
Prepared datasets for 1 couriers

=== Step 2: Training Personalized Models ===


Training: 100%|██████████████████████████████████████████████████████████████████████████| 1/1 [00:27<00:00, 27.29s/it]

Courier 35: Early stop at epoch 123 (val_loss=0.8545)
Courier 35: Trained 123 epochs, Val Loss=0.8211
Trained 1 models

=== Step 3: Saving Results ===
Models saved to: courier_models


({35: {'path': 'courier_models\\courier_35_model.pth',
   'n_samples': 1600,
   'best_val_loss': np.float64(0.8210617661476135),
   'epochs_trained': 123,
   'scaler_mean': array([1.48739007e+00, 5.57426470e+01, 3.75891074e+01, 1.79142842e+00,
          2.78114888e+00, 4.99144842e-01, 4.90192852e-01, 1.37900725e+01,
          2.76167663e+02, 3.43720000e+01, 4.77857143e-02]),
   'scaler_scale': array([5.83303505e+00, 1.18883784e-01, 1.35249201e-01, 2.80428573e+00,
          4.42159668e+00, 2.75618903e-02, 3.14167901e-02, 7.96276315e+00,
          3.00647079e+02, 1.90886777e+01, 1.55747686e-01])}},
 {35: {'X': array([[-0.25488805,  0.08890606,  0.22546827, ..., -0.63709212,
           -0.22903629, -0.30681492],
          [-0.2548573 ,  0.10343977,  0.24590622, ..., -0.51399097,
           -0.22903629, -0.30681492],
          [-0.25344702,  0.09050912,  0.07500624, ..., -0.68397248,
           -1.01484243, -0.30681492],
          ...,
          [ 1.66317468,  1.72506212, -1.72041538, ...,

In [81]:
import torch
import os

def load_courier_model(courier_id, checkpoint_dir='courier_models', device='cpu'):
    """
    Load pre-trained courier model from .pth checkpoint
    
    Args:
        courier_id: ID of courier to load
        checkpoint_dir: Directory with saved models
        device: 'cpu' or 'cuda'
    
    Returns:
        model: Loaded PolygonAlgNN model (in eval mode)
        metadata: Training metadata (n_samples, val_loss, etc.)
    """
    # Load model checkpoint
    model_path = os.path.join(checkpoint_dir, f'courier_{courier_id}_model.pth')
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")
    
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    
    # Initialize model with correct input_dim
    model = PolygonAlgNN(input_dim=checkpoint['input_dim'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    metadata = {
        'n_samples': checkpoint.get('n_samples', 0),
        'val_loss': checkpoint.get('best_val_loss', 0),
        'input_dim': checkpoint['input_dim'],
        'scaler_mean': checkpoint['scaler_mean'],
        'scaler_scale': checkpoint['scaler_scale']
    }
    
    print(f"Loaded courier {courier_id} model ({metadata['n_samples']} samples, val_loss={metadata['val_loss']:.4f})")
    
    return model, metadata

def evaluate_policy_quality(model, X_val, y_val, device='cpu'):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_val, dtype=torch.float).to(device)
        y_tensor = torch.tensor(y_val, dtype=torch.float).to(device)
        
        logits = model(X_tensor)
        preds = F.softmax(logits, dim=1)
        
        # Metric 1: Mean Squared Error
        mse = torch.mean((preds - y_tensor) ** 2).item()
        
        # Metric 2: Per-algorithm correlation
        correlations = []
        for i in range(4):
            corr = np.corrcoef(preds[:, i].cpu().numpy(), y_tensor[:, i].cpu().numpy())[0, 1]
            correlations.append(corr)
        
        # Metric 3: Top-1 accuracy
        pred_top = torch.argmax(preds, dim=1).cpu().numpy()
        y_top = np.argmax(y_val, axis=1)
        top1_acc = np.mean(pred_top == y_top)
        
        return {
            'policy': preds,
            'mse': mse,
            'correlations': correlations,
            'top1_accuracy': top1_acc,
            'policy_sums': preds.sum(dim=1).mean().item()
        }

In [82]:
test_results = {}

for courier_id, items in trainer.items():
    # Load THIS courier's model
    model, metadata = load_courier_model(courier_id)
    
    # Extract features ONLY for THIS courier's polygons
    courier_features = []
    courier_strats = []
    polygon_ids = []
    
    global_strats = items['EPOS_local_strat']
    for polygon_id in items['polygon_list']:
        if polygon_id not in properties_global:
            continue
        
        props = properties_global[polygon_id]
        params = AUG_global_map[polygon_id]
        avg_dist = np.mean([np.mean(row) for row in params['distance matrix']])
        
        features = [
            props['area_km2'],
            props['centroid_lat'],
            props['centroid_lon'],
            props['bounding_box']['width_km'],
            props['bounding_box']['height_km'],
            props['relative_position']['normalized_lat'],
            props['relative_position']['normalized_lon'],
            props['relative_position']['distance_from_center_km'],
            avg_dist,
            props['num_points'],
            params['cluster']/7.0
        ]
        
        opt_strategy = global_strats.get(polygon_id, items['EPOS_global_strat'])
        
        courier_features.append(features)
        courier_strats.append(opt_strategy)
        polygon_ids.append(polygon_id)
    
    # Test THIS model on THIS courier's data ONLY
    result_test = evaluate_policy_quality(model, courier_features, courier_strats)
    
    test_results[courier_id] = {
        'mse': result_test['mse'],
        'top1_acc': result_test['top1_accuracy'],
        'n_samples': len(courier_features),
        'correlations': result_test['correlations']
    }
    
    print(f"Courier {courier_id}: MSE={result_test['mse']:.4f}, "
          f"Top-1 Acc={result_test['top1_accuracy']:.2%}, "
          f"Samples={len(courier_features)}")

# Сразу график 

Loaded courier 35 model (1600 samples, val_loss=0.8211)
Courier 35: MSE=0.1302, Top-1 Acc=55.55%, Samples=2000


In [83]:
results = np.array(result_test['policy'])

C:\Users\Administrator\AppData\Local\Temp\ipykernel_14692\3993539883.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  results = np.array(result_test['policy'])


In [84]:
results

array([[0.05038173, 0.14147532, 0.760299  , 0.04784398],
       [0.0544415 , 0.13595833, 0.7603227 , 0.04927758],
       [0.04278797, 0.14386915, 0.7683647 , 0.0449781 ],
       ...,
       [0.07876607, 0.11623209, 0.7436613 , 0.06134053],
       [0.05417126, 0.13951562, 0.7572499 , 0.04906331],
       [0.07418343, 0.11968278, 0.747244  , 0.0588898 ]],
      shape=(2000, 4), dtype=float32)

In [85]:
results_mean = np.mean(results, axis = 0)

In [86]:
results_mean

array([0.05786988, 0.1328733 , 0.75774205, 0.05151545], dtype=float32)

In [87]:
global_strategies_test = []

for ite, value in global_strats.items():
    global_strategies_test.append(value)

global_strategies_test_arr = np.array(global_strategies_test)
np.mean(global_strategies_test_arr, axis = 0)

array([0.05463231, 0.21922627, 0.54586228, 0.18027914])

In [ ]:
def get_full_geomap(global_properties):
    lat_lon_list = []
    global_orders = []
    for polygon_id, props in global_properties.items():
        lat_lon_list.extend(props['points_cloud'])
        global_orders.extend(props['point_ids'])
    points_tree = cKDTree(lat_lon_list)

    return points_tree, global_orders

cached_tree, global_orders = get_full_geomap(properties)

def get_nearest_point(courier_id, cached_tree, global_orders, k=1):
    """Get k nearest polygons using KD-Tree"""
    lat, lon = trained_couriers[courier_id]['base_lat'], trained_couriers[courier_id]['base_lon']
    distances, indices = cached_tree.query([lat, lon], k=min(k, len(global_orders)))
    return global_orders[indices]

def prepare_logistics_basis_clean(polygon_id, courier_id, cached_tree, global_orders):
    """
    Cleaner approach: Build new matrix instead of modifying existing
    """
    # Get depot point
    mod_point = get_nearest_point(courier_id, cached_tree, global_orders, k=1)
    if isinstance(mod_point, np.ndarray):
        mod_point = mod_point[0]
    
    start_point = get_basic_order_ID(mod_point)
    
    # Get original data
    order_test = AUG_global_map[polygon_id]['full map'].copy()
    original_dist = AUG_global_map[polygon_id]['distance matrix']
    n = len(order_test)
    
    if start_point not in order_test:
        # New order list with depot at index 0
        orders_new = [mod_point] + order_test
        
        # Create (N+1)×(N+1) matrix
        dist_matrix = [[0] * (n + 1) for _ in range(n + 1)]
        
        # Fill depot row/column (index 0)
        for i in range(n+1):                    
            order_id = get_basic_order_ID(orders_new[i])
            dist_matrix[i][0] = int(get_distance(distance_array, idx_dict, start_point, order_id))
        for j in range(n+1):
            order_id = get_basic_order_ID(orders_new[j])
            dist_matrix[0][j] = int(get_distance(distance_array, idx_dict, order_id, start_point))
        for i in range(n):
            for j in range(n):
                dist_matrix[i+1][j+1] = original_dist[i][j]
        
        depot_idx = 0
    else:
        orders_new = order_test
        dist_matrix = original_dist
        depot_idx = order_test.index(start_point)
    
    return orders_new, depot_idx, dist_matrix

def evaluate_personalized_models(trained_models, properties, AUG_global_map, n_samples=1000):
    
    # Sample 1000 random polygons
    all_polygons = list(properties.keys())
    test_polygons = random.sample(all_polygons, min(n_samples, len(all_polygons)))    
    
    results = {
        'courier_id': [],
        'polygon_id': [],
        'chosen_algorithm': [],
        'W_i': [],
        'winning_reward': [],
        'predicted_policy': []
    }
    
    algorithms = ['christofides', 'branch_and_cut', 'lkh3', 'greedy']
    
    # For each courier
    for courier_id, model_data in trained_models.items():
        model = model_data['model']
        model.eval()
        price = 500
        
        for polygon_id in test_polygons:
            config = generate_polygon_config(polygon_id, properties, n_clusters=8)
            time_window = [0,43200]
            U = 900
            time_window_gap = time_window[1]-time_window[0]
            CT_SA = 2
            props = properties[polygon_id]
            params = AUG_global_map[polygon_id]
            avg_dist = np.mean([np.mean(dist_list) for dist_list in params['distance matrix']])
            features = [
                props['area_km2'],
                props['centroid_lat'],
                props['centroid_lon'],
                props['bounding_box']['width_km'],
                props['bounding_box']['height_km'],
                props['relative_position']['normalized_lat'],
                props['relative_position']['normalized_lon'],
                props['relative_position']['distance_from_center_km'],
                avg_dist,
                props['num_points']
            ]            
            
            # Predict policy
            with torch.no_grad():
                features_tensor = torch.tensor([features], dtype=torch.float)
                logits = model(features_tensor)
                policy = F.log_softmax(logits, dim=1).squeeze().numpy()
            
            # Sample algorithm from policy
            chosen_idx = np.random.choice(4, p=policy)
            chosen_algo = algorithms[chosen_idx]
            orders, start, dist_matrix = prepare_logistics_basis_clean(polygon_id, courier_id, cached_tree, global_orders)
            st = time.time()
            
            if chosen_algo == 'christofides':
                route, route_time = OptimizerInterface.solve_christofides(orders, start, dist_matrix)
                fin = time.time()
            elif chosen_algo == 'branch_and_cut':
                route, route_time = OptimizerInterface.solve_branch_and_cut(orders, start, dist_matrix)
                fin = time.time()
            elif chosen_algo == 'lkh3':
                route, route_time = OptimizerInterface.solve_lkh3(orders, start, dist_matrix)
                fin = time.time()
            else:
                route, route_time = OptimizerInterface.solve_greedy(orders, start, dist_matrix)
                fin = time.time()

            CT_i = fin-st
            W_i = (config['alpha']*(U - price)+
                   config['beta']*(time_window_gap-route_time)+
                   config['gamma']*(CT_SA-CT_i)
                  )
            
            results['courier_id'].append(courier_id)
            results['polygon_id'].append(polygon_id)
            results['chosen_algorithm'].append(chosen_algo)
            results['W_i'].append(W_i)
            results['predicted_policy'].append(policy.tolist())
    
    # Calculate winning rewards
    results_df = pd.DataFrame(results)
    results_df = results_df.copy()  # Ensure copy at creation
    for idx in range(len(results_df)):
        courier_id = results_df.loc[idx, 'courier_id']
        polygon_id = results_df.loc[idx, 'polygon_id']
        W_i = results_df.loc[idx, 'W_i']
        
        # Compare with other couriers on same polygon
        same_polygon = results_df[results_df['polygon_id'] == polygon_id]
        W_j_max = same_polygon[same_polygon['courier_id'] != courier_id]['W_i'].max()
        
        winning_reward = 1 if W_i > W_j_max else 0
        
        results_df.at[idx, 'winning_reward'] = winning_reward
    
    return results_df